# ArcGIS Online item description editor

**Welcome!**  
This notebook helps you scan, review, and update ArcGIS Online item Terms of Use HTML (the `licenseInfo` field) at scale.

**Where this notebook can run**  
- VS Code on macOS (local Jupyter kernel).
- VS Code on Windows (local Jupyter kernel).
- ArcGIS Online Notebook (JupyterLab-style).

**How to use this notebook**  
- Start with **1. Setup and authenticate** and run the code cell to connect to your organization.
- Run **2. Scan your content** to find matching `licenseInfo` terms.
- Save scan outputs, optionally run a secondary scan for additional terms, and review strict matches.
- Run a **dry run** to preview exactly what would change.
- Generate the side-by-side report to visually compare old and new HTML.
- Commit updates only after validating the dry-run/report outputs.

**Notes**  
- Some org-wide scans take time; progress messages are printed while users and folders are traversed.
- The workflow is designed to be safe-by-default: review first, then update.
- If you need to restart, use Kernel restart and rerun from setup.

**tldr;**

In [1]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does**  
- Authenticates to ArcGIS Online (multiple contexts)
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames:`scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user friendly side-by-side HTML review report for visual QA.
- Applies updates and edits the itemsonly after explicit confirmation, then exports success/error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))


**What this notebook does**  
- Authenticates to ArcGIS Online (multiple contexts)
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames:`scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user friendly side-by-side HTML review report for visual QA.
- Applies updates and edits the itemsonly after explicit confirmation, then exports success/error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.


## 1. Setup and authenticate

In [ ]:
# In the toolbar above, select View > Collapse All Code to hide the code.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
import ast
from pathlib import Path
import arcgis
import pandas as pd
import ipywidgets as widgets
from functools import partial

# Support local VS Code (macOS/Windows) and ArcGIS Online Notebook locations for helper_functions.py
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_HELPER_DIRS = [NOTEBOOK_DIR, Path("/arcgis/home")]
for p in CANDIDATE_HELPER_DIRS:
    helper_file = p / "helper_functions.py"
    if helper_file.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from helper_functions import (
    detect_environment,
    authenticate_gis,
    initialize_ui,
    scan_org_licenseinfo_without_10k_cap,
    build_licenseinfo_update_plan,
    build_side_by_side_report,
    show_dry_run,
    apply_licenseinfo_updates,
    OFFICIAL_TOU_HTML,
 )

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {"gis": None}

# Define local variables
matches_df = None
errors_df = None
all_items_df = None
TARGET_STRINGS = []

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")

output1 = initialize_ui("output")

def setup_notebook_btn(button):
    with output1:
        output1.clear_output()
        print("Setting up the notebook environment...")
        print(f"\tPython version: {sys.version}")
        print(f"\tArcGIS for Python API version: {arcgis.__version__}")
        authenticate_gis(context=context)
        if context.get("gis") is not None:
            globals()["gis"] = context["gis"]
            print("Authentication complete.")

# Create widgets
btn1 = initialize_ui("button", description="Setup Notebook", width="200px")
display(btn1)
btn1.on_click(setup_notebook_btn)
display(output1)

Initializing...
Currently running in VSCode Notebook environment


Button(description='Setup Notebook', layout=Layout(height='40px', width='200px'), style=ButtonStyle())

Output()

## 2. Scan your content
This cell scans your content for the text strings you input into the "Search strings:" box.

In [ ]:
# Cell 2: Define terms and scan org content for licenseInfo matches
output2 = initialize_ui("output")

txt2 = initialize_ui("label", value="Enter text/URLs to find in the Terms of Use section -->", width="350px")
txt2_2 = initialize_ui("label", value="Separate multiple terms with commas: ['term1', 'term2']", width="350px")
txt2_3 = initialize_ui("label", value="And enclose any spaces in single quotes: ['\"term with spaces\"', 'another term']", width="350px")
input2 = initialize_ui(
    "textarea",
    value='["https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png", "esrilogo"]',
    description="",
    width="500px",
    height="70px",
)
vbox2 = widgets.VBox([txt2, txt2_2, txt2_3])
hbox2 = initialize_ui("hbox", elements=[vbox2, input2])
btn2 = initialize_ui("button", description="Scan for items", width="200px")
display(widgets.VBox([hbox2, btn2, output2]))

def parse_target_terms(raw_text):
    text = (raw_text or "").strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    return [t.strip().strip('"').strip("'") for t in text.split(",") if t.strip()]

def run_primary_scan(button):
    global matches_df, errors_df, all_items_df, TARGET_STRINGS
    with output2:
        output2.clear_output()
        if context.get("gis") is None:
            print("Please run Setup and authenticate first.")
            return

        terms = parse_target_terms(input2.value)
        if not terms:
            print("No search terms provided.")
            return

        print(f"Running scan with {len(terms)} term(s)...")
        matches_df, errors_df, all_items_df = scan_org_licenseinfo_without_10k_cap(
            context["gis"],
            target_strings=terms,
        )
        TARGET_STRINGS = terms

        print(f"Matches: {len(matches_df)} | Errors: {len(errors_df)}")
        display(matches_df.head(20))

btn2.on_click(run_primary_scan)

## 3. Save the scan results to csv files
This cell saves the scan. You can change the filenames and location if you'd like

In [ ]:
# Cell 3: Save latest scan outputs to CSV
output3 = initialize_ui("output")
btn3 = initialize_ui("button", description="Save files")
display(widgets.VBox([btn3, output3]))

def save_scan_outputs(button):
    with output3:
        output3.clear_output()
        if "matches_df" not in globals() or "errors_df" not in globals():
            print("Run the scan first so matches_df and errors_df exist.")
            return
        matches_df.to_csv("scan_matches.csv", index=False)
        errors_df.to_csv("scan_errors.csv", index=False)
        all_items_df.to_csv("scan_all_items.csv", index=False)
        print("Saved: scan_matches.csv, scan_errors.csv, and scan_all_items.csv")

btn3.on_click(save_scan_outputs)

## 4. Optionally reload results from a previous scan
use csv

In [ ]:
matches_df = pd.read_csv("scan_matches.csv", dtype={"item_id": str})
errors_df = pd.read_csv("scan_errors.csv", dtype={"item_id": str})
all_items_df = pd.read_csv("scan_all_items.csv", dtype={"item_id": str})

## 4. Secondary scan
This cell saves you time if you want to search additional terms.
If you want to search again, click the SECONDARY_SCAN check box and add the new terms to NEW_TERMS and run the cell below

In [ ]:
# Cell 4: Optional secondary scan using new terms and excluding previous matches
output4 = initialize_ui("output")
checkbox4 = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
input4 = initialize_ui(
    "textarea",
    value='["https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg"]',
    description="New search strings:",
    width="500px",
    height="70px",
)

btn4 = initialize_ui("button", description="Run secondary scan")
display(widgets.VBox([checkbox4, input4, btn4, output4]))

def run_secondary_scan(button):
    with output4:
        output4.clear_output()

        if not checkbox4.value:
            print("Secondary scan not run (checkbox is unchecked).")
            return

        if context.get("gis") is None:
            print("Please run Setup and authenticate first.")
            return

        if "matches_df" in globals() and not matches_df.empty:
            exclude_ids = set(matches_df["item_id"].dropna().astype(str))
        elif Path("scan_matches.csv").exists():
            previous_matches_df = pd.read_csv("scan_matches.csv", dtype={"item_id": str})
            exclude_ids = set(previous_matches_df["item_id"].dropna().astype(str))
        else:
            exclude_ids = set()

        new_terms = parse_target_terms(input4.value)
        if not new_terms:
            print("No new search terms provided.")
            return

        print(f"Running secondary scan with {len(new_terms)} term(s)...")
        new_matches_df, new_errors_df, new_all_items_df = scan_org_licenseinfo_without_10k_cap(
            context["gis"],
            target_strings=new_terms,
            exclude_item_ids=exclude_ids,
        )

        if not new_matches_df.empty and exclude_ids:
            new_matches_df = new_matches_df[~new_matches_df["item_id"].isin(exclude_ids)].copy()

        new_matches_df.to_csv("secondary_scan_matches.csv", index=False)

        globals()["new_matches_df"] = new_matches_df
        globals()["new_errors_df"] = new_errors_df
        globals()["new_all_items_df"] = new_all_items_df

        print(f"New matches: {len(new_matches_df)} | Errors: {len(new_errors_df)}")
        display(new_matches_df.head(20))

btn4.on_click(run_secondary_scan)

## 4. Strict matching output

In [ ]:
# If you need strict matching on the exact term you can filter the scan results to a new DataFrame.
exact_url_df = matches_df[
    matches_df["matched_terms"].str.contains(
        "https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
        case=False,
        na=False
    )
]
# Print the exact results
print(f"Number of items with exact matches: {len(exact_url_df)}")
exact_url_df

## 6. Dry run

In [ ]:
# Cell 6. Do a dry run before making any changes
output6 = initialize_ui("output")

btn6 = initialize_ui("button", description="Dry Run", width='150px', height='40px')
display(widgets.VBox([btn6, output6]))
btn6.on_click(partial(build_licenseinfo_update_plan, output6=output6, context=context))

plan_df = build_licenseinfo_update_plan(matches_df, OFFICIAL_TOU_HTML)
dry_run_table = show_dry_run(plan_df, max_rows=200)
dry_run_table

## 7. Export dry run results

In [ ]:
# Optionally export the update results to a CSV file for record-keeping and review.
plan_df.to_csv("ToU_update_plan_dry_run.csv", index=False)

## 8. Create report

In [ ]:
# Generate a HTML report for easy comparison of old vs new ToU before updating items.
report_path = build_side_by_side_report(
    plan_df,
    out_html="ToU_side_by_side_report.html",
    only_updates=True,
    max_rows=200
)

## 9. Commit updates

In [ ]:
# Execute the updates, but only after explicit confirmation input to avoid accidental changes.
# If you'd like to change the phrase wording, edit the require_phrase parameter below.
success_df, update_errors_df = apply_licenseinfo_updates(
    gis,
    plan_df,
    require_phrase="APPLY UPDATES",
    pause_seconds=0.0
)
# As confirmation of successful updates, display the first few successfully updated items
success_df.head()

## 10. Export final results

In [ ]:
# Optionally export the update results to CSV files for record-keeping
success_df.to_csv("update_ToU_successes.csv", index=False)
update_errors_df.to_csv("update_ToU_errors.csv", index=False)